# IF29 - Projet de classification de profils X/Twitter
## Notebook 01 : Exploration initiale du dataset Tweet_Worldcup

### Objectif : comprendre la structure de la base MongoDB,
identifier les champs disponibles et planifier l'agrégation.

In [2]:
from pymongo import MongoClient
import pandas as pd
import json
from pprint import pprint

# --- 1. Connexion à MongoDB ---
# Adaptez le nom de la base et de la collection à ce que vous voyez dans Compass
client = MongoClient("mongodb://localhost:27017/")
print("Bases disponibles :", client.list_database_names())

db = client["Tweet_Worldcup"]
print("Collections :", db.list_collection_names())

col = db["tweets"]

ModuleNotFoundError: No module named 'pymongo'

In [ ]:
# --- 2. Volumétrie ---
n_tweets = col.estimated_document_count()
print(f"Nombre total de tweets (estimation rapide) : {n_tweets:,}")

# Nombre d'utilisateurs distincts (peut être long, ~quelques minutes sur 27 Go)
pipeline_users = [
    {"$group": {"_id": "$user.id"}},
    {"$count": "n_users"}
]
res = list(col.aggregate(pipeline_users, allowDiskUse=True))
print(f"Nombre d'utilisateurs uniques : {res[0]['n_users']:,}")

Nombre total de tweets (estimation rapide) : 4,569,999
Nombre d'utilisateurs uniques : 1,843,439


In [ ]:
# --- 3. Inspection d'un document type ---
# Un tweet brut peut contenir 30+ champs imbriqués
sample = col.find_one()
print("Champs racine :", list(sample.keys()))
print("\nChamps de l'objet user :", list(sample.get("user", {}).keys()))
print("\nChamps de entities :", list(sample.get("entities", {}).keys()))

# Affichage complet d'un document (utile pour repérer ce qui nous intéresse)
print(json.dumps(sample, default=str, indent=2)[:3000])

Champs racine : ['_id', 'created_date', 'current_time', 'in_reply_to_status_id_str', 'in_reply_to_status_id', 'created_at', 'in_reply_to_user_id_str', 'source', 'retweeted_status', 'retweet_count', 'retweeted', 'geo', 'filter_level', 'in_reply_to_screen_name', 'is_quote_status', 'id_str', 'in_reply_to_user_id', 'favorite_count', 'id', 'text', 'place', 'lang', 'quote_count', 'favorited', 'coordinates', 'truncated', 'timestamp_ms', 'reply_count', 'entities', 'contributors', 'user']

Champs de l'objet user : ['utc_offset', 'friends_count', 'profile_image_url_https', 'listed_count', 'profile_background_image_url', 'default_profile_image', 'favourites_count', 'description', 'created_at', 'is_translator', 'profile_background_image_url_https', 'protected', 'screen_name', 'id_str', 'profile_link_color', 'translator_type', 'id', 'geo_enabled', 'profile_background_color', 'lang', 'profile_sidebar_border_color', 'profile_text_color', 'verified', 'profile_image_url', 'time_zone', 'url', 'contribut

In [ ]:
# --- 4. Couverture temporelle ---
pipeline_time = [
    {"$group": {
        "_id": None,
        "min_date": {"$min": "$created_at"},
        "max_date": {"$max": "$created_at"}
    }}
]
print(list(col.aggregate(pipeline_time)))

[{'_id': None, 'min_date': 'Fri Jun 15 00:00:00 +0000 2018', 'max_date': 'Thu Jun 14 23:59:59 +0000 2018'}]


In [ ]:
# --- 5. Distribution du nombre de tweets par utilisateur ---
# Cela conditionne la qualité des features agrégées :
# un utilisateur avec 1 seul tweet ne donnera pas un comportement fiable.
pipeline_dist = [
    {"$group": {"_id": "$user.id", "n_tweets": {"$sum": 1}}},
    {"$bucket": {
        "groupBy": "$n_tweets",
        "boundaries": [1, 2, 5, 10, 50, 100, 500, 1000, 100000],
        "default": "other",
        "output": {"count": {"$sum": 1}}
    }}
]
dist = list(col.aggregate(pipeline_dist, allowDiskUse=True))
pd.DataFrame(dist)

NameError: name 'col' is not defined